# DUY TAN UNIVERSITY - CS467 ADVANCED ARTIFICIAL INTELLIGENCE
## Assignment 15 - Individual Mini-Project: NLP & Recommendation Systems

**Student Name:** Lê Văn Hoài Rin  
**Student ID:** 29211332113  
**Class / Section:** CS467C (No. 14)  
**Assigned Project Code:** MP-C14  
**Project Title:** Volunteer and Community Service Recommender  
**Domain:** NGO and Student Civic Engagement  
**GitHub Repository:** [cs467-miniproject-29211332113](https://github.com/Rynne25/cs467-miniproject-29211332113)

---
### Project Overview & Scenario
University students want to join volunteer activities that fit their skills, schedule, and causes they care about.
This mini-project builds a volunteer recommender using **Natural Language Processing (NLP)** on activity descriptions and past student volunteer feedback, combining:
1. **Popularity Baseline**: Bayesian-weighted average ratings.
2. **NLP Content-Based Recommender**: TF-IDF vectorization & Cosine Similarity for user profiles & free-text queries.
3. **User-Based Collaborative Filtering (User-CF)**: Consensus from like-minded volunteers.
4. **Weighted Hybrid Recommender**: Smooth blending with automated cold-start fallback.

### 1. Setup & Environment

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to module path
sys.path.insert(0, os.path.abspath(".."))

from src.nlp_pipeline import VolunteerNLPPipeline, clean_text
from src.recommender import (
    PopularityRecommender,
    ContentBasedRecommender,
    UserBasedCollaborativeFiltering,
    HybridRecommender
)
from src.evaluate import evaluate_models, plot_visualizations, run_test_cases

sns.set_theme(style="whitegrid")
print("All modules imported successfully!")

### 2. Dataset Exploration (data/items.csv & data/ratings.csv)

In [ ]:
df_items = pd.read_csv("../data/items.csv")
df_ratings = pd.read_csv("../data/ratings.csv")

print(f"Items Shape: {df_items.shape} | Columns: {list(df_items.columns)}")
print(f"Ratings Shape: {df_ratings.shape} | Columns: {list(df_ratings.columns)}")

n_users = df_ratings["user_id"].nunique()
n_items = df_items["item_id"].nunique()
n_ratings = len(df_ratings)
sparsity = (1.0 - n_ratings / (n_users * n_items)) * 100

print(f"Users: {n_users} | Items: {n_items} | Interactions: {n_ratings}")
print(f"User-Item Matrix Sparsity: {sparsity:.1f}% missing entries (Realistic sparse interaction matrix)")
df_items.head(5)

### 3. Natural Language Processing (NLP) Pipeline & Feature Extraction

In [ ]:
nlp = VolunteerNLPPipeline(max_features=1200, ngram_range=(1, 2))
item_vectors = nlp.fit_items(df_items)

print(f"TF-IDF Matrix Shape: {item_vectors.shape}")
print(f"Fitted Vocabulary Size: {len(nlp.feature_names)}")
print(f"Sample vocabulary terms: {list(nlp.feature_names[:15])}")

### 4. Recommendation Engines Initialization

In [ ]:
# 1. Popularity Baseline
pop_rec = PopularityRecommender(min_ratings_threshold=2)
pop_rec.fit(df_ratings, df_items)

# 2. NLP Content-Based
cb_rec = ContentBasedRecommender(nlp)
cb_rec.fit(df_items)

# 3. User-Based Collaborative Filtering
cf_rec = UserBasedCollaborativeFiltering(k_neighbors=5)
cf_rec.fit(df_ratings, df_items)

# 4. Weighted Hybrid Recommender
hybrid_rec = HybridRecommender(cb_rec, cf_rec, alpha=0.55)

print("All four recommender models trained and ready!")

### 5. Mandatory Test Cases Verification
Demonstrating the 3 required test cases specified in the project requirements:

In [ ]:
print("=" * 75)
print("TEST CASE 1: Natural Language Interest Query Search")
print("=" * 75)
query = "weekend beach cleanup and environmental education for kids"
recs_tc1 = cb_rec.recommend_for_query(query, top_n=5)

for r, item in enumerate(recs_tc1, 1):
    print(f"[{r}] [{item['item_id']}] {item['activity_title']}")
    print(f"    Cause Area : {item['cause_area']}")
    print(f"    Similarity : {item['score']:.4f}")
    print(f"    Top Terms  : {', '.join(item['matching_keywords'])}\n")

In [ ]:
print("=" * 75)
print("TEST CASE 2: Returning Volunteer Recommendation (USR_02)")
print("Volunteer USR_02 previously rated tutoring & mentoring programs 5.0/5.0")
print("=" * 75)

recs_tc2 = hybrid_rec.recommend("USR_02", df_ratings, df_items, top_n=5)
for r, item in enumerate(recs_tc2, 1):
    print(f"[{r}] [{item['item_id']}] {item['activity_title']}")
    print(f"    Cause Area   : {item['cause_area']}")
    print(f"    Hybrid Score : {item['score']:.4f} ({item['method']})")
    if item.get('matching_keywords'):
        print(f"    Overlap      : {', '.join(item['matching_keywords'])}")
    print()

In [ ]:
print("=" * 75)
print("TEST CASE 3: Cold-Start Item Retrieval (ITEM_038 with ZERO ratings)")
print("=" * 75)
cold_id = "ITEM_038"
cold_ratings_count = (df_ratings["item_id"] == cold_id).sum()
print(f"Ratings in dataset for {cold_id}: {cold_ratings_count} (Cold start verified)")

cold_query = "rural community donation drive collecting children books and toys"
recs_tc3 = cb_rec.recommend_for_query(cold_query, top_n=5)

for r, item in enumerate(recs_tc3, 1):
    is_target = " <--- [MATCHED COLD-START ITEM!]" if item['item_id'] == cold_id else ""
    print(f"[{r}] [{item['item_id']}] {item['activity_title']}{is_target}")
    print(f"    Cause Area : {item['cause_area']}")
    print(f"    Similarity : {item['score']:.4f}")
    print(f"    Top Terms  : {', '.join(item['matching_keywords'])}\n")

### 6. Offline Evaluation & Performance Visualization

In [ ]:
metrics_df, df_train, df_test = evaluate_models(df_items, df_ratings, test_ratio=0.20)
print("\n--- OFFLINE METRICS COMPARISON (Holdout 20% per user) ---")
display(metrics_df)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
df_plot = metrics_df.melt(id_vars="Method", value_vars=["Precision@5", "Recall@5", "HitRate@5"], var_name="Metric", value_name="Score")
sns.barplot(data=df_plot, x="Method", y="Score", hue="Metric", palette="mako", ax=ax)
ax.set_title("Performance Comparison Across Recommendation Approaches", fontsize=13, fontweight="bold")
ax.set_ylim(0, 1.05)
for p in ax.patches:
    val = p.get_height()
    if val > 0:
        ax.annotate(f"{val:.2f}", (p.get_x() + p.get_width() / 2., val),
                    ha="center", va="baseline", fontsize=9, xytext=(0, 4), textcoords="offset points")
plt.tight_layout()
plt.show()